# Naive Bayes

In [2]:
# Load the SMS Spam Collection dataset from Hugging Face Datasets
from datasets import load_dataset
ds = load_dataset("ucirvine/sms_spam")

d:\ML_Phd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

# Convert the dataset to a pandas DataFrame for easier manipulation
df = pd.DataFrame(ds["train"])
# Display information about the DataFrame
print(df.info)


<bound method DataFrame.info of                                                     sms  label
0     Go until jurong point, crazy.. Available only ...      0
1                       Ok lar... Joking wif u oni...\n      0
2     Free entry in 2 a wkly comp to win FA Cup fina...      1
3     U dun say so early hor... U c already then say...      0
4     Nah I don't think he goes to usf, he lives aro...      0
...                                                 ...    ...
5569  This is the 2nd time we have tried 2 contact u...      1
5570             Will ü b going to esplanade fr home?\n      0
5571  Pity, * was in mood for that. So...any other s...      0
5572  The guy did some bitching but I acted like i'd...      0
5573                       Rofl. Its true to its name\n      0

[5574 rows x 2 columns]>


In [4]:
# Display statistical summary of the DataFrame
print(df.describe())

             label
count  5574.000000
mean      0.134015
std       0.340699
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000


In [5]:
# Split the dataset into features and labels
X = df["sms"]  # Features (SMS messages)
y = df["label"]  # Labels (spam or ham)

In [6]:
# Print the features and target labels

print("Features: \n", X.head())
print("Target labels: \n", y.head())

Features: 
 0    Go until jurong point, crazy.. Available only ...
1                      Ok lar... Joking wif u oni...\n
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: sms, dtype: str
Target labels: 
 0    0
1    0
2    1
3    0
4    0
Name: label, dtype: int64


In [7]:
# Print target unique labels
print(y.unique())

[0 1]


In [8]:
# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=1)

In [9]:
# Print all test and train data
print("X_train: \n", X_train)
print("X_test: \n", X_test)
print("y_train: \n", y_train)
print("y_test: \n", y_test)

X_train: 
 4351    Yes. Rent is very expensive so its the way we ...
524                    That's very rude, you on campus?\n
714     Save yourself the stress. If the person has a ...
5036    How many times i told in the stage all use to ...
686                      Have you finished work yet? :)\n
                              ...                        
905     We're all getting worried over here, derek and...
5192    Our records indicate u maybe entitled to 5000 ...
3980    CERI U REBEL! SWEET DREAMZ ME LITTLE BUDDY!! C...
235     Text & meet someone sexy today. U can find a d...
5157    MY NEW YEARS EVE WAS OK. I WENT TO A PARTY WIT...
Name: sms, Length: 3623, dtype: str
X_test: 
 1447    Looks like u wil b getting a headstart im leav...
2032    I noe la... U wana pei bf oso rite... K lor, o...
4432     2mro i am not coming to gym machan. Goodnight.\n
4888    Todays Vodafone numbers ending with 4882 are s...
5276    Hi. Hope ur day * good! Back from walk, table ...
               

## A Really Dumb Spam Filter (one word)

Let's start with just one word: "lottery".

- **S** = the event "the message is spam"
- **L** = the event "the message contains the word *lottery*"

We want P(S | L), the probability that a message is spam given that it contains "lottery". Bayes's Theorem says:

$$P(S \mid L) = \frac{P(L \mid S)\,P(S)}{P(L \mid S)\,P(S) + P(L \mid \neg S)\,P(\neg S)}$$

In simple words: out of all the messages that contain "lottery", what share are spam?

If we have many old messages already marked spam / not spam, we can easily count:
- **P(L | S)**: what share of **spam** messages contain "lottery"
- **P(L | ¬S)**: what share of **normal** messages contain "lottery"

### Example
Say **40%** of spam SMS contain "lottery", but only **1%** of normal SMS do.

## A More Sophisticated Spam Filter (many words)

Now we have a vocabulary of many words $w_1, w_2,\dot, w_n$. For each word we estimate:
- P(word | spam): how often a spam message contains it
- P(word | not spam): how often a normal message contains it

### The "naive" assumption 
To combine many words easily, Naive Bayes makes a big assumption:

- Given that a message is spam (or not spam), the words appear independently of each other.

- Then the probability of a whole set of words is simply the product of the single-word probabilities:

$$P(X_1 = x_1, \dots, X_n = x_n \mid S) = P(X_1 = x_1 \mid S) \times \dots \times P(X_n = x_n \mid S)$$

### Why is it called "naive"?
Because this assumption is often wrong! Example:
- Half of all spam is about a "lottery" win, and the other half is about a "KYC" update.
- These two words never appear together in the same spam.
- But Naive Bayes thinks: P(lottery | spam) = 0.5, P(KYC | spam) = 0.5, so P(both | spam) = 0.5 × 0.5 = 0.25. The true answer is 0!

In [10]:
def spam_prob_one_word(p_word_if_spam, p_word_if_not_spam, p_spam=0.5):
    """Bayes's Theorem for one word"""
    p_not_spam = 1 - p_spam
    top = p_word_if_spam * p_spam
    return top / (top + p_word_if_not_spam * p_not_spam)

#Example: "lottery" appears in 40% of spam messages and 1% of normal messages
# Assume a message is equally likely to be spam or not (p_spam = 0.5)
print(f"If spam and normal messages are equally common: P(spam | 'lottery') = {spam_prob_one_word(0.40, 0.01):.1%}")

# More realistic: only 20% of all SMS are spam
print(f"If only 20% of all SMS are spam:  P(spam | 'lottery') = {spam_prob_one_word(0.40, 0.01, p_spam=0.2):.1%}")

#If lottery and kyc appear in spam messages independently.
p_lottery_if_spam = 0.5
p_kyc_if_spam = 0.5
print(f"Naive Bayes estimate of P(lottery AND kyc | spam): {p_lottery_if_spam * p_kyc_if_spam:.1%}")
print(f"True value (they never appear together): 0.0")

If spam and normal messages are equally common: P(spam | 'lottery') = 97.6%
If only 20% of all SMS are spam:  P(spam | 'lottery') = 90.9%
Naive Bayes estimate of P(lottery AND kyc | spam): 25.0%
True value (they never appear together): 0.0


## Implementation, Step by Step

We build the classifier from 4 small functions:

| Step | Function | What it does |
|---|---|---|
| 1 | `tokenize(message)` | Break a message into its distinct words |
| 2 | `count_words(training_set)` | For each word: how many spam and non-spam messages contain it |
| 3 | `word_probabilities(...)` | Turn counts into smoothed probabilities |
| 4 | `spam_probability(...)` | Combine all words (using logs) $\to$ P(spam) for a new message |

In [11]:
import re
def tokenize(message):
    """Tokenizes a string into words, removing punctuation and converting to lowercase."""
    # Remove punctuation and split by whitespace
    tokens = re.findall(r'\b\w+\b', message.lower())
    return tokens

In [12]:
from collections import defaultdict

def count_words(training_set):
    """Training set is a list of tuples (text, label)"""
    counts = defaultdict(lambda: [0, 0])  # word -> [count_in_spam, count_not_in_spam]
    for text, label in training_set:
        words = tokenize(text)
        for word in words:
            if label == 'spam':
                counts[word][0] += 1
            else:
                counts[word][1] += 1
    return counts

In [13]:
def word_probabilities(counts, total_spam, total_non_spam, k=0.5):
    """Turns the word counts into a list of triplets (word, P(word|spam), P(word|not spam))"""
    return [(w, (spam + k) / (total_spam + 2 * k), (non_spam + k) / (total_non_spam + 2 * k))
            for w, (spam, non_spam) in counts.items()]

In [14]:
import math


def spam_probability(word_probs, message):
    """Given a list of word probabilities and a message, returns P(spam|message)"""
    message_words = tokenize(message)
    log_prob_if_spam = log_prob_if_not_spam = 0.0

    for word, prob_if_spam, prob_if_not_spam in word_probs:
        if word in message_words:
            log_prob_if_spam += math.log(prob_if_spam)
            log_prob_if_not_spam += math.log(prob_if_not_spam)
        else:
            log_prob_if_spam += math.log(1.0 - prob_if_spam)
            log_prob_if_not_spam += math.log(1.0 - prob_if_not_spam)

    prob_if_spam = math.exp(log_prob_if_spam)
    prob_if_not_spam = math.exp(log_prob_if_not_spam)

    return prob_if_spam / (prob_if_spam + prob_if_not_spam)

In [15]:
class NaiveBayesClassifier:
    def __init__(self, k=0.5):
        self.k = k
        self.word_probs = []

    def train(self, training_set):
        """Training set is a list of tuples (text, label)"""
        num_spam = len([1 for _, label in training_set if label == 'spam'])
        num_non_spam = len(training_set) - num_spam

        counts = count_words(training_set)
        self.word_probs = word_probabilities(counts, num_spam, num_non_spam, self.k)

    def classify(self, message):
        """Returns the probability that the message is spam"""
        return spam_probability(self.word_probs, message)

In [16]:
# Example usage
training_set = list(zip(X_train, y_train))
test_set = list(zip(X_test, y_test))
classifier = NaiveBayesClassifier(k=0.5)
classifier.train(training_set)
# Classify each test message and store (message, is_spam)
classified = [
    (message, classifier.classify(message) >= 0.5)
    for message, _ in test_set
]

In [17]:
from collections import Counter

# Create a Counter instance
counts = Counter(
    (actual_label == 1, predicted_spam)
    for (_, actual_label), (_, predicted_spam) in zip(test_set, classified)
)

In [19]:
classified.sort(key=lambda x: classifier.classify(x[0]))
spammiest_hams = list(filter(lambda x: not x[1], classified))[-5:]  # Get the 5 hams that were most confidently classified as spam
hammiest_spams = list(filter(lambda x: x[1], classified))[:5] # Get the 5 spams that were most confidently classified as ham 

In [20]:
def p_spam_given_word(word_probs):
    """Uses Bayes's theorem to compute the probability of spam given a word"""

    word, prob_if_spam, prob_if_not_spam = word_probs
    return prob_if_spam / (prob_if_spam + prob_if_not_spam)

words = sorted(classifier.word_probs, key=p_spam_given_word)

some_spammiest_words = words[-5:]  # Get the 5 words most indicative of spam    
some_hammiest_words = words[:5]  # Get the 5 words most indicative of ham

In [21]:
print("5 words most indicative of spam:")
for word, prob_if_spam, prob_if_not_spam in some_spammiest_words:
    print(f"{word}: P(spam|word) = {p_spam_given_word((word, prob_if_spam, prob_if_not_spam)):.1%}")

5 words most indicative of spam:
dreamz: P(spam|word) = 99.9%
buddy: P(spam|word) = 99.9%
blokes: P(spam|word) = 99.9%
thirtyeight: P(spam|word) = 99.9%
boyfriend: P(spam|word) = 99.9%


In [22]:
print("5 words most indicative of spam:")
for word, prob_if_spam, prob_if_not_spam in some_spammiest_words:
    print(f"{word}: P(spam|word) = {p_spam_given_word((word, prob_if_spam, prob_if_not_spam)):.1%}")    

5 words most indicative of spam:
dreamz: P(spam|word) = 99.9%
buddy: P(spam|word) = 99.9%
blokes: P(spam|word) = 99.9%
thirtyeight: P(spam|word) = 99.9%
boyfriend: P(spam|word) = 99.9%


In [23]:
print("Filtering out the 5 words most indicative of ham:")

for word, prob_if_spam, prob_if_not_spam in some_hammiest_words:
    print(f"{word}: P(spam|word) = {p_spam_given_word((word, prob_if_spam, prob_if_not_spam)):.1%}")

Filtering out the 5 words most indicative of ham:
i: P(spam|word) = 48.1%
you: P(spam|word) = 55.1%
to: P(spam|word) = 55.4%
a: P(spam|word) = 66.0%
the: P(spam|word) = 68.2%


In [24]:
print("Filtering out the 5 words most indicative of ham:")
for word, prob_if_spam, prob_if_not_spam in some_hammiest_words:
    print(f"{word}: P(spam|word) = {p_spam_given_word((word, prob_if_spam, prob_if_not_spam)):.1%}")

Filtering out the 5 words most indicative of ham:
i: P(spam|word) = 48.1%
you: P(spam|word) = 55.1%
to: P(spam|word) = 55.4%
a: P(spam|word) = 66.0%
the: P(spam|word) = 68.2%
